In [ ]:
import os
os.environ["OPENCV_LOG_LEVEL"] = "OFF"
import cv2
import torch
import numpy as np
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim.lr_scheduler import ReduceLROnPlateau
import albumentations as A
from albumentations.pytorch import ToTensorV2
import matplotlib.pyplot as plt
from tqdm import tqdm
from scipy import ndimage
    

CONFIG = {
    "IMAGE_SIZE": 384,
    "BATCH_SIZE": 8,
    "DEVICE": "cuda" if torch.cuda.is_available() else "cpu",
    "LR": 3e-4,
    "EPOCHS": 50,
    "PATIENCE": 5
}

In [ ]:
class RoadsDataset(Dataset):
    def __init__(self, image_dir, mask_dir, transform=None):
        self.image_dir = image_dir
        self.mask_dir = mask_dir
        self.transform = transform
        self.images = os.listdir(image_dir)

    def __len__(self):
        return len(self.images)

    def __getitem__(self, index):
        img_name_full = self.images[index]
        img_name_no_ext = os.path.splitext(img_name_full)[0]
        
        img_path = os.path.join(self.image_dir, img_name_full)
        
        mask_path = None
        for ext in ['.tif', '.tiff', '.png']:
            possible_path = os.path.join(self.mask_dir, img_name_no_ext + ext)
            if os.path.exists(possible_path):
                mask_path = possible_path
                break
        
        image = cv2.imread(img_path)
        mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE) if mask_path else None
        
        if image is None:
            raise FileNotFoundError(f"Картинка не найдена: {img_path}")
        if mask is None:
            raise FileNotFoundError(f"Маска не найдена для {img_name_full} в {self.mask_dir}")

        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        mask = np.float32(mask > 128)

        if self.transform:
            augmentations = self.transform(image=image, mask=mask)
            image = augmentations["image"]
            mask = augmentations["mask"]

        return image, mask.unsqueeze(0)

In [ ]:
import segmentation_models_pytorch as smp

model = smp.Unet(
    encoder_name="vgg19",
    encoder_weights="imagenet",
    in_channels=3,
    classes=1,
    activation=None
).to(CONFIG["DEVICE"])

In [ ]:
train_transform = A.Compose([
    A.RandomCrop(height=CONFIG["IMAGE_SIZE"], width=CONFIG["IMAGE_SIZE"]), 
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.RandomRotate90(p=0.5),
    A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ToTensorV2(),
])

val_transform = A.Compose([
    A.CenterCrop(height=CONFIG["IMAGE_SIZE"], width=CONFIG["IMAGE_SIZE"]), 
    A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ToTensorV2(),
])

dice_loss = smp.losses.DiceLoss(mode='binary')
focal_loss = smp.losses.FocalLoss(mode='binary')

def loss_fn(preds, targets):
    return dice_loss(preds, targets) + focal_loss(preds, targets)

optimizer = torch.optim.AdamW(model.parameters(), lr=CONFIG["LR"], weight_decay=1e-4)

In [ ]:
TRAIN_IMG_DIR = "data/tiff/train"
TRAIN_MASK_DIR = "data/tiff/train_labels"
VAL_IMG_DIR = "data/tiff/val"
VAL_MASK_DIR = "data/tiff/val_labels"
TEST_IMG_DIR = "data/tiff/test"
TEST_MASK_DIR = "data/tiff/test_labels"

train_ds = RoadsDataset(image_dir=TRAIN_IMG_DIR, mask_dir=TRAIN_MASK_DIR, transform=train_transform)
val_ds = RoadsDataset(image_dir=VAL_IMG_DIR, mask_dir=VAL_MASK_DIR, transform=val_transform)
test_ds = RoadsDataset(image_dir=TEST_IMG_DIR, mask_dir=TEST_MASK_DIR, transform=val_transform)

train_loader = DataLoader(train_ds, batch_size=CONFIG["BATCH_SIZE"], shuffle=True, 
                          num_workers=1, pin_memory=True, drop_last=True)
val_loader = DataLoader(val_ds, batch_size=CONFIG["BATCH_SIZE"], shuffle=False, 
                        num_workers=0, pin_memory=True)
test_loader = DataLoader(test_ds, batch_size=CONFIG["BATCH_SIZE"], shuffle=False, num_workers=0)

scheduler = ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=3)

In [ ]:
def check_accuracy(loader, model, device="cuda"):
    model.eval()
    dice_score = 0
    
    with torch.no_grad():
        for x, y in loader:
            x = x.to(device)
            y = y.to(device)
            
            preds = torch.sigmoid(model(x))
            preds = (preds > 0.5).float()
            
            intersection = (preds * y).sum()
            union = (preds + y).sum()
            
            dice_score += (2. * intersection) / (union + 1e-8)
    
    avg_dice = dice_score / len(loader)
    print(f"Val Dice Score: {avg_dice:.4f}")
    
    model.train()
    return avg_dice.item()

def denormalize(img_tensor, mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]):
    img = img_tensor.cpu().permute(1, 2, 0).numpy()
    img = img * np.array(std) + np.array(mean)
    return np.clip(img, 0, 1)

def save_predictions_as_imgs(loader, model, folder="saved_images/", device="cuda"):
    model.eval()
    if not os.path.exists(folder):
        os.makedirs(folder)

    for idx, (x, y) in enumerate(loader):
        x = x.to(device)
        with torch.no_grad():
            preds = torch.sigmoid(model(x))
            preds = (preds > 0.5).float()
        
        plt.figure(figsize=(10, 5))
        plt.subplot(1, 3, 1)
        plt.title("Input Image")
        plt.imshow(denormalize(x[0]))
        plt.subplot(1, 3, 2)
        plt.title("Ground Truth")
        plt.imshow(y[0].cpu().squeeze(), cmap="gray")
        plt.subplot(1, 3, 3)
        plt.title("Prediction")
        plt.imshow(preds[0].cpu().squeeze(), cmap="gray")
        plt.show()
        break 
    model.train()

In [ ]:
def train_fn_simple(loader, model, optimizer, loss_fn):
    loop = tqdm(loader)
    model.train()
    total_loss = 0

    for batch_idx, (data, targets) in enumerate(loop):
        data = data.to(CONFIG["DEVICE"])
        targets = targets.float().to(CONFIG["DEVICE"])

        predictions = model(data)
        loss = loss_fn(predictions, targets)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        loop.set_postfix(loss=loss.item())
    
    return total_loss / len(loader)

best_dice = 0
patience_counter = 0

for epoch in range(CONFIG["EPOCHS"]):
    print(f"\n--- Epoch {epoch+1}/{CONFIG['EPOCHS']} ---")
    
    avg_loss = train_fn_simple(train_loader, model, optimizer, loss_fn)
    
    current_dice = check_accuracy(val_loader, model, device=CONFIG["DEVICE"])
    
    scheduler.step(current_dice)
    current_lr = optimizer.param_groups[0]['lr']
    print(f"Current Learning Rate: {current_lr:.6f}")   

    if (epoch + 1) % 5 == 0:
        save_predictions_as_imgs(val_loader, model, device=CONFIG["DEVICE"])
    
    if current_dice > best_dice:
        best_dice = current_dice
        patience_counter = 0
        torch.save({
            "state_dict": model.state_dict(),
            "optimizer": optimizer.state_dict(),
            "epoch": epoch,
            "best_dice": best_dice
        }, "best_model.pth.tar")
        print(f"Saved new best model! Dice: {best_dice:.4f}")
    else:
        patience_counter += 1
        print(f"⏳ Patience: {patience_counter}/{CONFIG['PATIENCE']}")
        
        if patience_counter >= CONFIG["PATIENCE"]:
            print("Early stopping triggered!")
            break

print(f"\nTraining finished! Best Val Dice: {best_dice:.4f}")

In [ ]:
from sklearn.metrics import precision_score, recall_score

def evaluate_test_set(loader, model, device="cuda"):
    model.eval()
    metrics = {"dice": [], "iou": [], "precision": [], "recall": []}
    
    with torch.no_grad():
        for x, y in tqdm(loader, desc="Testing"):
            x, y = x.to(device), y.to(device)
            preds = torch.sigmoid(model(x))
            preds = (preds > 0.5).float()

            y_true = y.cpu().numpy().flatten()
            y_pred = preds.cpu().numpy().flatten()

            intersection = (preds * y).sum().item()
            union = (preds + y).sum().item()
            
            dice = (2. * intersection) / (union + 1e-8)
            iou = intersection / (union - intersection + 1e-8)
            prec = precision_score(y_true, y_pred, zero_division=0)
            rec = recall_score(y_true, y_pred, zero_division=0)

            metrics["dice"].append(dice)
            metrics["iou"].append(iou)
            metrics["precision"].append(prec)
            metrics["recall"].append(rec)

    print("\n--- Final Test Metrics ---")
    for k, v in metrics.items():
        print(f"{k.upper()}: {np.mean(v):.4f}")
    return metrics

In [ ]:
def plot_learning_curves(train_losses, val_dices):
    plt.figure(figsize=(12, 5))
    
    plt.subplot(1, 2, 1)
    plt.plot(train_losses, label='Train Loss')
    plt.title('Loss Evolution')
    plt.xlabel('Epoch')
    plt.legend()

    plt.subplot(1, 2, 2)
    plt.plot(val_dices, label='Val Dice', color='orange')
    plt.title('Dice Score Evolution')
    plt.xlabel('Epoch')
    plt.legend()
    
    plt.show()

In [ ]:
def visualize_results(loader, model, device="cuda", n_images=5):
    model.eval()
    images_shown = 0
    plt.figure(figsize=(15, n_images * 4))
    
    with torch.no_grad():
        for x, y in loader:
            if images_shown >= n_images: break
            
            x_dev = x.to(device)
            preds = torch.sigmoid(model(x_dev))
            preds = (preds > 0.5).float().cpu()

            for i in range(x.shape[0]):
                if images_shown >= n_images: break
                
                plt.subplot(n_images, 3, images_shown * 3 + 1)
                plt.imshow(denormalize(x[i]))
                plt.title("Original")
                
                plt.subplot(n_images, 3, images_shown * 3 + 2)
                plt.imshow(y[i].squeeze(), cmap='gray')
                plt.title("Ground Truth")
                
                plt.subplot(n_images, 3, images_shown * 3 + 3)
                plt.imshow(preds[i].squeeze(), cmap='gray')
                plt.title("Prediction")
                
                images_shown += 1
    plt.tight_layout()
    plt.show()

In [ ]:
checkpoint = torch.load("best_model.pth.tar")
model.load_state_dict(checkpoint["state_dict"])

test_metrics = evaluate_test_set(test_loader, model, device=CONFIG["DEVICE"])
visualize_results(test_loader, model, device=CONFIG["DEVICE"])